# Plan A: smoke test

Proves the whole pipeline end to end on 20 dummy rows: install, download the English checkpoint, dry-run, train 1 epoch, reload, predict, save to Drive, and (later, on your PC) load it on CPU through `laya_adapter`.
**This tests plumbing only. The numbers mean nothing about quality.**

How to use it: run the cells **top to bottom, one at a time**. Each code cell starts with a comment saying what you should see. If something differs, paste the output back to Claude. After a kernel restart, re-run cell 1 first (it holds the settings), then continue from where you were.

Nothing here overwrites the original checkpoint: training reads a downloaded copy in `/content/work/base-english` and writes to a new folder.

In [ ]:
# CELL 1: settings + GPU check. Re-run this cell first after any kernel restart.
# EXPECT: a line like "NVIDIA L4, 23034 MiB" (about 23 GB), then "cuda True", then the run id.
import datetime
import hashlib
import json
import os
import queue
import re
import shutil
import subprocess
import sys
import threading
import time

import torch

BASE_REPO = "convaiinnovations/laya"
BASE_REV = "7b928d828b7b0e022f929d9bd2e44165aa270148"      # the snapshot used locally (laya-test/hf_home)
LAYA_COMMIT = "3cf26cbcb18725dbc2d127bb8bb2c4c43243ae63"   # laya v0.4.0
# sha256 of the LOCAL copy of the base files: cell 4 proves Colab downloaded identical ones
LOCAL_BASE_SHA256 = {
    "model.safetensors": "891102d372688fc2a094dac56a384bc537b87c63f21f9f3dac0be2b7cbc8d86c",
    "rl_agent_config.json": "ae287b56bbcf5f8c4f4541ae9dfd00c914c4c48b940b8398c3058af37ba92bbd",
}

DRIVE = "/content/drive/MyDrive/laya-finetuning"
WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)

# one run id per session, kept on disk so re-running this cell does not change it
_rid = f"{WORK}/run_id.txt"
if not os.path.exists(_rid):
    open(_rid, "w").write("smoke-" + datetime.datetime.now().strftime("%Y%m%d-%H%M"))
RUN_ID = open(_rid).read().strip()

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
print("cuda", torch.cuda.is_available(), "| torch", torch.__version__)
print("run id:", RUN_ID)

In [ ]:
# CELL 2: mount Google Drive and create the folders.
# EXPECT: Colab asks you to allow Drive access, then "Mounted at /content/drive" and "DRIVE_OK [...]".
# If it prints DRIVE_MOUNT_FAILED, stop and paste the message (fallback: a private Hugging Face repo).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    for sub in ("checkpoints", "data", "runs"):
        os.makedirs(f"{DRIVE}/{sub}", exist_ok=True)
    print("DRIVE_OK", sorted(os.listdir(DRIVE)))
except Exception as e:
    print("DRIVE_MOUNT_FAILED:", repr(e))
    raise

In [ ]:
# CELL 3: install laya (same commit as the local v0.4.0 checkout) and the same transformers as local.
# EXPECT: no errors, then "laya 0.4.0" and "transformers 5.16.1".
# If pip says a restart is needed: restart the kernel, re-run cell 1, SKIP this cell, continue at cell 4.
!pip -q install "laya @ git+https://github.com/NandhaKishorM/laya.git@{LAYA_COMMIT}" "transformers==5.16.1"
from importlib.metadata import version
print("laya", version("laya"), "| transformers", version("transformers"), "| torch", version("torch"))

In [ ]:
# CELL 4: download the English checkpoint at the pinned revision and compare it with the local copy.
# EXPECT: files listed (model.safetensors about 842 MB), "weights identical to local copy: True",
# "config identical to local copy: True", and max_len 512 / head_max_len 192.
from huggingface_hub import snapshot_download


def sha256_file(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


BASE = snapshot_download(BASE_REPO, revision=BASE_REV, local_dir=f"{WORK}/base-english",
                         allow_patterns=["model.safetensors", "rl_agent_config.json", "encoder/*", "tokenizer/*"])
for dp, _, fns in sorted(os.walk(BASE)):
    for fn in sorted(fns):
        p = os.path.join(dp, fn)
        if "/.cache" not in p.replace("\\", "/"):
            print(f"{os.path.getsize(p) / 1e6:9.1f} MB  {os.path.relpath(p, BASE)}")

BASE_SHA256 = {name: sha256_file(f"{BASE}/{name}") for name in LOCAL_BASE_SHA256}
same_w = BASE_SHA256["model.safetensors"] == LOCAL_BASE_SHA256["model.safetensors"]
same_c = BASE_SHA256["rl_agent_config.json"] == LOCAL_BASE_SHA256["rl_agent_config.json"]
print("weights identical to local copy:", same_w)
print("config identical to local copy:", same_c)
BASE_CFG = json.load(open(f"{BASE}/rl_agent_config.json"))
print("max_len", BASE_CFG["max_len"], "| head_max_len", BASE_CFG["head_max_len"], "| encoder", BASE_CFG["encoder"])
assert same_w and same_c, "downloaded base differs from the local snapshot: stop and tell Claude"

In [ ]:
# CELL 5: the real production questions, frozen from qa-analysis LAYA_CHECKS (tools/export_checks.py).
# EXPECT: "checks sha OK: ac9a5f3c019a..." and "9 questions" followed by the check names.
FROZEN = json.loads(r"""{
  "frozen": {
    "checks": {
      "empathy": {
        "yes_means": "PASS",
        "pass_threshold": 0.78,
        "question": {
          "type": "noul",
          "instructions": "Does the agent acknowledge the customer's frustration with empathy?",
          "criteria": {
            "true": "the agent acknowledges the customer's frustration or feelings and responds with care",
            "false": "the agent ignores, brushes off or is rude about the customer's frustration"
          },
          "labels": {
            "true": "A",
            "false": "B"
          }
        }
      },
      "active_listening": {
        "yes_means": "FAIL",
        "pass_threshold": 0.19,
        "question": {
          "type": "noul",
          "instructions": "Does the second agent question ask for the same information as the first?",
          "criteria": {
            "true": "the agent repeats the same question and asks again for information already requested",
            "false": "the second question asks for something different from the first"
          }
        }
      },
      "ownership": {
        "yes_means": "FAIL",
        "pass_threshold": 0.52,
        "question": {
          "type": "noul",
          "instructions": "Does the agent deflect responsibility, blame another team, or refuse to help?",
          "criteria": {
            "true": "the agent deflects, blames another team or department, refuses to help, or tells the customer to contact someone else",
            "false": "the agent takes ownership and keeps working on the customer's problem"
          },
          "labels": {
            "true": "A",
            "false": "B"
          }
        }
      },
      "rapport": {
        "yes_means": "FAIL",
        "pass_threshold": 0.81,
        "question": {
          "type": "noul",
          "instructions": "Is the agent rude, condescending, dismissive, sarcastic or unprofessional?",
          "criteria": {
            "true": "the agent is rude, condescending, dismissive, sarcastic or unprofessional toward the customer",
            "false": "the agent is polite, respectful and professional"
          }
        }
      },
      "probing": {
        "yes_means": "PASS",
        "pass_threshold": 0.6,
        "question": {
          "type": "noul",
          "instructions": "Did the agent ask at least one genuine diagnostic question about the customer's problem?",
          "criteria": {
            "true": "the agent asked at least one question that investigates the cause or specifics of the problem, such as when it started, which device, what error or lights, or what was already tried",
            "false": "the agent only asked greetings, identity verification, confirmations, upsell, scheduling or closing questions"
          },
          "labels": {
            "true": "A",
            "false": "B"
          }
        }
      },
      "resolved": {
        "yes_means": "PASS",
        "pass_threshold": 0.73,
        "question": {
          "type": "noul",
          "instructions": "Did the agent explicitly confirm that the customer's issue was resolved?",
          "criteria": {
            "true": "the agent explicitly confirms the issue is resolved or fixed, usually after checking with the customer",
            "false": "the agent never confirms resolution: the call just ends, the agent assumes it is fixed, the customer says it still fails, or it is handed off"
          },
          "labels": {
            "true": "A",
            "false": "B"
          }
        }
      },
      "escalation": {
        "yes_means": "FAIL",
        "pass_threshold": 0.47,
        "question": {
          "type": "noul",
          "instructions": "Did the customer ask for a supervisor, manager or escalation that the agent refused, ignored or discouraged?",
          "criteria": {
            "true": "the customer asked for a supervisor, manager or escalation and the agent refused, ignored, discouraged or faked it",
            "false": "the customer did not ask to escalate, or the agent handled the request properly"
          }
        }
      },
      "hostility": {
        "yes_means": "FAIL",
        "pass_threshold": 0.79,
        "question": {
          "type": "noul",
          "instructions": "Is the agent hostile, abusive, insulting, threatening, mocking, or hangs up on the customer?",
          "criteria": {
            "true": "the agent insults, mocks, threatens, is abusive toward the customer, or abruptly hangs up",
            "false": "the agent stays professional and respectful, even if the customer is angry"
          }
        }
      },
      "paraphrasing": {
        "yes_means": "PASS",
        "pass_threshold": 0.5,
        "question": {
          "type": "noul",
          "instructions": "Does the agent restate or paraphrase the customer's problem back to them?",
          "criteria": {
            "true": "the agent restates the customer's problem in their own words to confirm they understood it",
            "false": "the agent never restates the customer's problem"
          }
        }
      }
    },
    "caps": {
      "EMPATHY_MAX_SNIPPETS": 2,
      "PROBING_PROBLEM_MAX_WORDS": 50,
      "PROBING_QUESTIONS_MAX_WORDS": 100,
      "RESOLVED_TAIL_LINES": 10,
      "PARAPHRASE_CUSTOMER_MAX_WORDS": 50,
      "PARAPHRASE_AGENT_MAX_WORDS": 150
    }
  },
  "sha256": "ac9a5f3c019a68ba07b88b0aa37ae160100a1315fe24812cfbc0123e449ec6b7",
  "n_questions": 9,
  "qa_analysis_commit": "378f3e5217dd2d591f3afbc044bfbc1d888690d9",
  "qa_analysis_branch": "version-10.2-laya-finetune",
  "qa_analysis_dirty": true
}""")


def canonical_sha(obj):
    text = json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


assert canonical_sha(FROZEN["frozen"]) == FROZEN["sha256"], "frozen checks were altered"
CHECKS = FROZEN["frozen"]["checks"]
print("checks sha OK:", FROZEN["sha256"][:12] + "...", "| from qa-analysis commit", FROZEN["qa_analysis_commit"][:10])
print(len(CHECKS), "questions:", ", ".join(CHECKS))

In [ ]:
# CELL 6: write smoke.jsonl, 20 hand-written rows (2 per check, plus 2 "replay" rows with soft targets).
# Each row asks the REAL frozen question; the cell checks that. One row has two questions, so 20 rows = 21 questions.
# EXPECT: "wrote 20 rows, 21 questions", "all check rows use the frozen production question", and a small table.
SMOKE_JSONL = r"""{"state": "Agent: You should have restarted the router before you called, that's the first thing anyone should do.\nAgent: That's basic stuff.", "questions": {"q": {"type": "noul", "instructions": "Is the agent rude, condescending, dismissive, sarcastic or unprofessional?", "criteria": {"true": "the agent is rude, condescending, dismissive, sarcastic or unprofessional toward the customer", "false": "the agent is polite, respectful and professional"}}}, "expected": {"q": true}, "meta": {"id": "rapport_fail", "check": "rapport", "verdict": "FAIL"}}
{"state": "Agent: Obviously I want to get this sorted for you today, so let me check the line.\nAgent: As I said earlier, I'll stay on the line with you until it's back up.", "questions": {"q": {"type": "noul", "instructions": "Is the agent rude, condescending, dismissive, sarcastic or unprofessional?", "criteria": {"true": "the agent is rude, condescending, dismissive, sarcastic or unprofessional toward the customer", "false": "the agent is polite, respectful and professional"}}}, "expected": {"q": false}, "meta": {"id": "rapport_pass", "check": "rapport", "verdict": "PASS"}}
{"state": "CUSTOMER: My internet keeps dropping every evening around eight and the router light turns orange. I have a client presentation tomorrow.\nAGENT: Let me make sure I have this right: the connection drops each evening around eight, the router goes orange, and you need it stable before tomorrow's presentation. Is that correct?", "questions": {"q": {"type": "noul", "instructions": "Does the agent restate or paraphrase the customer's problem back to them?", "criteria": {"true": "the agent restates the customer's problem in their own words to confirm they understood it", "false": "the agent never restates the customer's problem"}}}, "expected": {"q": true}, "meta": {"id": "para_pass", "check": "paraphrasing", "verdict": "PASS"}}
{"state": "CUSTOMER: My internet keeps dropping every evening around eight and the router light turns orange. I have a client presentation tomorrow.\nAGENT: Which light is blinking orange? What time did the evening drop start? Is the router in the living room?", "questions": {"q": {"type": "noul", "instructions": "Does the agent restate or paraphrase the customer's problem back to them?", "criteria": {"true": "the agent restates the customer's problem in their own words to confirm they understood it", "false": "the agent never restates the customer's problem"}}}, "expected": {"q": false}, "meta": {"id": "para_fail", "check": "paraphrasing", "verdict": "FAIL"}}
{"state": "Turn 6: What colour is the light on the front of the modem?\n... Turn 14: Can you tell me the colour of the light on the modem?\nCustomer: I just said, it's red.", "questions": {"q": {"type": "noul", "instructions": "Does the second agent question ask for the same information as the first?", "criteria": {"true": "the agent repeats the same question and asks again for information already requested", "false": "the second question asks for something different from the first"}}}, "expected": {"q": true}, "meta": {"id": "al_fail", "check": "active_listening", "verdict": "FAIL"}}
{"state": "Turn 6: What colour is the light on the front of the modem?\n... Turn 22: What is the light doing now after the restart?", "questions": {"q": {"type": "noul", "instructions": "Does the second agent question ask for the same information as the first?", "criteria": {"true": "the agent repeats the same question and asks again for information already requested", "false": "the second question asks for something different from the first"}}}, "expected": {"q": false}, "meta": {"id": "al_pass", "check": "active_listening", "verdict": "PASS"}}
{"state": "Agent: Let's test it together. Can you open a page for me?\nCustomer: It loaded right away, and the video call is stable too.\nAgent: Wonderful. So to confirm, the internet issue is fully resolved on your end?\nCustomer: Yes, completely resolved, thank you.\nAgent: Is there anything else I can help with?\nCustomer: No, that's all.", "questions": {"q": {"type": "noul", "instructions": "Did the agent explicitly confirm that the customer's issue was resolved?", "criteria": {"true": "the agent explicitly confirms the issue is resolved or fixed, usually after checking with the customer", "false": "the agent never confirms resolution: the call just ends, the agent assumes it is fixed, the customer says it still fails, or it is handed off"}, "labels": {"true": "A", "false": "B"}}}, "expected": {"q": true}, "meta": {"id": "resolved_pass", "check": "resolved", "verdict": "PASS"}}
{"state": "Agent: I've reset the line from here, it should be fine now.\nCustomer: Okay, I'll see how it goes.\nAgent: Thank you for calling S-Net. Have a good day.\nCustomer: Bye.", "questions": {"q": {"type": "noul", "instructions": "Did the agent explicitly confirm that the customer's issue was resolved?", "criteria": {"true": "the agent explicitly confirms the issue is resolved or fixed, usually after checking with the customer", "false": "the agent never confirms resolution: the call just ends, the agent assumes it is fixed, the customer says it still fails, or it is handed off"}, "labels": {"true": "A", "false": "B"}}}, "expected": {"q": false}, "meta": {"id": "resolved_fail", "check": "resolved", "verdict": "FAIL"}}
{"state": "Customer: This is the third time this week my connection died during a client call, I'm honestly about to cancel.\nAgent: Three drops in a week during client calls is really stressful, and I'm sorry you've been through that. Let's get to the bottom of it right now.\nCustomer: Thank you.\n", "questions": {"q": {"type": "noul", "instructions": "Does the agent acknowledge the customer's frustration with empathy?", "criteria": {"true": "the agent acknowledges the customer's frustration or feelings and responds with care", "false": "the agent ignores, brushes off or is rude about the customer's frustration"}, "labels": {"true": "A", "false": "B"}}}, "expected": {"q": true}, "meta": {"id": "empathy_pass", "check": "empathy", "verdict": "PASS"}}
{"state": "Customer: This is the third time this week my connection died during a client call, I'm honestly about to cancel.\nAgent: They can help with that. What's your account number?\nCustomer: It's 55 21 08.\n", "questions": {"q": {"type": "noul", "instructions": "Does the agent acknowledge the customer's frustration with empathy?", "criteria": {"true": "the agent acknowledges the customer's frustration or feelings and responds with care", "false": "the agent ignores, brushes off or is rude about the customer's frustration"}, "labels": {"true": "A", "false": "B"}}}, "expected": {"q": false}, "meta": {"id": "empathy_fail", "check": "empathy", "verdict": "FAIL"}}
{"state": "Agent: That's not my department, you'd have to call the billing team back tomorrow.", "questions": {"q": {"type": "noul", "instructions": "Does the agent deflect responsibility, blame another team, or refuse to help?", "criteria": {"true": "the agent deflects, blames another team or department, refuses to help, or tells the customer to contact someone else", "false": "the agent takes ownership and keeps working on the customer's problem"}, "labels": {"true": "A", "false": "B"}}}, "expected": {"q": true}, "meta": {"id": "ownership_fail", "check": "ownership", "verdict": "FAIL"}}
{"state": "Agent: That sits with another team's system, but I'll open the ticket with them myself so you don't have to call back.", "questions": {"q": {"type": "noul", "instructions": "Does the agent deflect responsibility, blame another team, or refuse to help?", "criteria": {"true": "the agent deflects, blames another team or department, refuses to help, or tells the customer to contact someone else", "false": "the agent takes ownership and keeps working on the customer's problem"}, "labels": {"true": "A", "false": "B"}}}, "expected": {"q": false}, "meta": {"id": "ownership_pass", "check": "ownership", "verdict": "PASS"}}
{"state": "CUSTOMER PROBLEM: My internet keeps dropping every evening around eight.\n\nThe agent asked these questions during the call:\n- Which lights on the router change colour when it drops?\n- Does it happen on every device or only your laptop?\n- Have you tried restarting the router?", "questions": {"q": {"type": "noul", "instructions": "Did the agent ask at least one genuine diagnostic question about the customer's problem?", "criteria": {"true": "the agent asked at least one question that investigates the cause or specifics of the problem, such as when it started, which device, what error or lights, or what was already tried", "false": "the agent only asked greetings, identity verification, confirmations, upsell, scheduling or closing questions"}, "labels": {"true": "A", "false": "B"}}}, "expected": {"q": true}, "meta": {"id": "probing_pass", "check": "probing", "verdict": "PASS"}}
{"state": "CUSTOMER PROBLEM: My internet keeps dropping every evening around eight.\n\nThe agent asked these questions during the call:\n- Is there anything else I can help you with today?\n- Would you like a short survey after the call?", "questions": {"q": {"type": "noul", "instructions": "Did the agent ask at least one genuine diagnostic question about the customer's problem?", "criteria": {"true": "the agent asked at least one question that investigates the cause or specifics of the problem, such as when it started, which device, what error or lights, or what was already tried", "false": "the agent only asked greetings, identity verification, confirmations, upsell, scheduling or closing questions"}, "labels": {"true": "A", "false": "B"}}}, "expected": {"q": false}, "meta": {"id": "probing_fail", "check": "probing", "verdict": "FAIL"}}
{"state": "Customer: My router has been dead since this morning.\nAgent: Maybe if you read the instructions you wouldn't be this clueless. I don't have time for this.\nCustomer: Excuse me?\nAgent: Whatever. Call someone else.", "questions": {"q": {"type": "noul", "instructions": "Is the agent hostile, abusive, insulting, threatening, mocking, or hangs up on the customer?", "criteria": {"true": "the agent insults, mocks, threatens, is abusive toward the customer, or abruptly hangs up", "false": "the agent stays professional and respectful, even if the customer is angry"}}}, "expected": {"q": true}, "meta": {"id": "hostility_fail", "check": "hostility", "verdict": "FAIL"}}
{"state": "Customer: You people are useless, my router has been dead since this morning!\nAgent: I hear how upsetting this is, and I'm going to fix it with you. Let me look at the line.\nCustomer: Fine.", "questions": {"q": {"type": "noul", "instructions": "Is the agent hostile, abusive, insulting, threatening, mocking, or hangs up on the customer?", "criteria": {"true": "the agent insults, mocks, threatens, is abusive toward the customer, or abruptly hangs up", "false": "the agent stays professional and respectful, even if the customer is angry"}}}, "expected": {"q": false}, "meta": {"id": "hostility_pass", "check": "hostility", "verdict": "PASS"}}
{"state": "Customer: I want to speak to a supervisor right now.\nAgent: There's no need for that, I can handle it. Supervisors can't do more than I can.\nCustomer: I asked for a supervisor.\nAgent: Let's just continue with the troubleshooting.", "questions": {"q": {"type": "noul", "instructions": "Did the customer ask for a supervisor, manager or escalation that the agent refused, ignored or discouraged?", "criteria": {"true": "the customer asked for a supervisor, manager or escalation and the agent refused, ignored, discouraged or faked it", "false": "the customer did not ask to escalate, or the agent handled the request properly"}}}, "expected": {"q": true}, "meta": {"id": "escalation_fail", "check": "escalation", "verdict": "FAIL"}}
{"state": "Customer: Can I speak to a supervisor?\nAgent: Of course. I'll transfer you now and pass on the full details so you don't repeat yourself.\nCustomer: Thanks.", "questions": {"q": {"type": "noul", "instructions": "Did the customer ask for a supervisor, manager or escalation that the agent refused, ignored or discouraged?", "criteria": {"true": "the customer asked for a supervisor, manager or escalation and the agent refused, ignored, discouraged or faked it", "false": "the customer did not ask to escalate, or the agent handled the request properly"}}}, "expected": {"q": false}, "meta": {"id": "escalation_pass", "check": "escalation", "verdict": "PASS"}}
{"state": "Agent: If we get cut off, I'll call you back on the number ending 4417 within ten minutes.\nCustomer: Great.", "questions": {"item_0": {"type": "noul", "instructions": "Did the agent do this: Offered a callback number?", "criteria": {"true": "the agent clearly did it: offered a callback number", "false": "the agent did not do it"}}}, "gold": {"item_0": {"probabilities": {"false": 0.2, "true": 0.8}}}, "meta": {"id": "replay_callback"}}
{"state": "Agent: I can't compare us with other providers, but here is what our plan includes. Before you go, would you be willing to rate this call?\nCustomer: Sure.", "questions": {"item_0": {"type": "noul", "instructions": "Does the agent violate this rule: Never discuss competitors?", "criteria": {"true": "an agent line clearly violates it: never discuss competitors", "false": "no agent line violates it"}}, "item_1": {"type": "noul", "instructions": "Did the agent do this: Offer a survey?", "criteria": {"true": "the agent clearly did it: offer a survey", "false": "the agent did not do it"}}}, "gold": {"item_0": {"probabilities": {"false": 0.85, "true": 0.15}}, "item_1": {"probabilities": {"false": 0.1, "true": 0.9}}}, "meta": {"id": "replay_batch"}}
"""
SMOKE = f"{WORK}/smoke.jsonl"
with open(SMOKE, "w", encoding="utf-8", newline="\n") as f:
    f.write(SMOKE_JSONL)
ROWS = [json.loads(line) for line in SMOKE_JSONL.splitlines() if line.strip()]

for r in ROWS:
    check = r["meta"].get("check")
    if check:
        assert r["questions"]["q"] == CHECKS[check]["question"], f"{r['meta']['id']} does not use the frozen question"
n_q = sum(len(r["questions"]) for r in ROWS)
print(f"wrote {len(ROWS)} rows, {n_q} questions to {SMOKE}")
print("all check rows use the frozen production question")
for r in ROWS:
    label = r["expected"]["q"] if "expected" in r else "gold"
    print(f"  {r['meta']['id']:<16} expected(yes)={label}")
SMOKE_SHA256 = hashlib.sha256(SMOKE_JSONL.encode("utf-8")).hexdigest()

In [ ]:
# CELL 7: --dry-run. Builds the training items with the real tokenizer; loads no weights, trains nothing.
# EXPECT: "dry-run: 20 rows read, 21 valid items, skipped: {}" (skipped must be empty).
!python -m laya.train_cli --data {SMOKE} --base {BASE} --dry-run --max-len 512 --head-max-len 192

In [ ]:
# CELL 8: baseline. The untouched base model answers all 21 questions; probabilities saved for later comparison.
# EXPECT: "device cuda" style load, then 21 numbers between 0 and 1 and "saved smoke_preds_base.json".
import laya


def predict_all(predict, rows):
    """{"<row id>/<question id>": P(yes)} using predict(state, questions) -> laya result."""
    out = {}
    for r in rows:
        res = predict(r["state"], r["questions"])
        for qid in r["questions"]:
            out[f"{r['meta']['id']}/{qid}"] = float(res["answers"][qid]["noul"])
    return out


base_agent = laya.load(BASE)
P_BASE = predict_all(lambda s, q: base_agent.predict(s, q), ROWS)
json.dump(P_BASE, open(f"{WORK}/smoke_preds_base.json", "w"), indent=1)
for k, v in P_BASE.items():
    print(f"  {k:<28} P(yes)={v:.3f}")
print("saved smoke_preds_base.json")
del base_agent
torch.cuda.empty_cache()

In [12]:
# CELL 9a: helpers for safe copying to Drive and for running laya-train with live output.
# Also writes laya_train_cuda.py: laya 0.4.0's trainer crashes on a GPU ("tensors on the same device ...") because it
# evaluates before moving the model to the GPU. This wrapper does that move first; the package itself is not edited.
# EXPECT: no output other than "helpers ready".
LAUNCHER = f"{WORK}/laya_train_cuda.py"
with open(LAUNCHER, "w", encoding="utf-8", newline="\n") as f:
    f.write(r'''"""laya-train with one fix for GPUs. Same arguments as `laya-train` / `python -m laya.train_cli`.

Why it exists: in laya 0.4.0, finetune() evaluates the calibration slice (or the --eval file) BEFORE training,
but load_checkpoint() returns the model on the CPU and only train_model() moves it to the GPU. On CUDA that
evaluation crashes with "Expected all tensors to be on the same device ... index_select". Laya's own tests run
on CPU, so they never hit it. This wrapper moves the model to the target device before every
calibration_records() call. Nothing in the installed laya package is edited.

Use it exactly like laya-train:
    python laya_train_cuda.py --data train.jsonl --base <dir> --out <dir> --loss soft-ce --epochs 4 ...
Drop this wrapper if a later laya release fixes it upstream (run once without it to find out).
"""
import sys

import laya.train as _train

_original = _train.calibration_records


def _calibration_records_on_device(model, tok, items, device, *args, **kwargs):
    model.to(device)      # a no-op when it is already there (after training)
    return _original(model, tok, items, device, *args, **kwargs)


# finetune() looks calibration_records up in laya.train's globals at call time, so this takes effect
_train.calibration_records = _calibration_records_on_device

from laya.train_cli import main  # noqa: E402  (imported after the patch on purpose)

if __name__ == "__main__":
    sys.exit(main())
''')

EPOCH_RE = re.compile(r"epoch (\d+)/(\d+) mean loss")


class SourceChanged(Exception):
    pass


class CopyMismatch(Exception):
    pass


def tree_stats(path, ignore=()):
    """Sorted (relative path, size, mtime_ns) for every file under path. Files that vanish mid-scan are skipped."""
    out = []
    for dp, dns, fns in os.walk(path):
        if dp == path:
            dns[:] = [d for d in dns if d not in ignore]
        for fn in fns:
            p = os.path.join(dp, fn)
            try:
                st = os.stat(p)
            except FileNotFoundError:
                continue
            out.append((os.path.relpath(p, path), st.st_size, st.st_mtime_ns))
    return sorted(out)


def wait_stable(path, quiet_s=30, poll_s=3, timeout_s=1800, newer_than=None, ignore=()):
    """Return once no file under path changed size or mtime for quiet_s seconds.

    With newer_than (a time.time() value), also require that something was written at or after it, so an
    older, already-stable copy is never mistaken for the checkpoint that is about to be written.
    """
    t0, last, since = time.time(), None, time.time()
    while time.time() - t0 < timeout_s:
        snap = tree_stats(path, ignore)
        if snap != last:
            last, since = snap, time.time()
        fresh = newer_than is None or (snap and max(m for _, _, m in snap) / 1e9 >= newer_than)
        if snap and fresh and time.time() - since >= quiet_s:
            return snap
        time.sleep(poll_s)
    raise TimeoutError(f"{path} was not stable within {timeout_s}s")


def copy_verified(src, dst, tag, ignore=()):
    """Copy src to a temp folder next to dst, verify it, then rename it into place.

    Raises SourceChanged if src changed while copying, CopyMismatch if the copy differs. dst is only
    replaced after both checks pass, so a half-written copy never sits under the final name.
    Returns (number of files, total bytes).
    """
    tmp = f"{dst}.tmp-{tag}"
    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    before = tree_stats(src, ignore)
    shutil.copytree(src, tmp, ignore=shutil.ignore_patterns(*ignore) if ignore else None)
    names_sizes = lambda stats: [(r, z) for r, z, _ in stats]
    if names_sizes(tree_stats(tmp)) != names_sizes(before):
        shutil.rmtree(tmp, ignore_errors=True)
        raise CopyMismatch(f"{tmp} differs from {src}")
    if tree_stats(src, ignore) != before:
        shutil.rmtree(tmp, ignore_errors=True)
        raise SourceChanged(src)
    old = None
    if os.path.exists(dst):
        old = f"{dst}.old-{tag}"
        os.rename(dst, old)
    os.rename(tmp, dst)
    if old:
        shutil.rmtree(old, ignore_errors=True)
    return len(before), sum(z for _, z, _ in before)


class CkptWatcher:
    """Copies src (checkpoint_latest/) to Drive each time an epoch finishes AND the folder has stopped changing.

    laya-train prints the "epoch N/M mean loss" line BEFORE it writes the checkpoint, so the line alone is
    not enough: after it, wait until nothing was written for quiet_s seconds (and something was written
    after the line), copy to a temp folder on Drive, verify, rename. finish() does one last pass after the
    process exits, because laya-train rewrites the config after calibration.
    Messages are queued and printed by the notebook's main thread (flush), never from this thread.
    """

    def __init__(self, src, dst, quiet_s=30):
        self.src, self.dst, self.quiet_s = src, dst, quiet_s
        self.work, self.msgs = queue.Queue(), queue.Queue()
        self.copies, self.errors = [], []
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def say(self, text):
        self.msgs.put(f"[watcher {time.strftime('%H:%M:%S')}] {text}")

    def feed(self, item):
        t, line = item
        m = EPOCH_RE.search(line)
        if m:
            self.work.put(("epoch", int(m.group(1)), t))

    def finish(self):
        self.work.put(("final", 0, time.time()))

    def stop(self):
        """Training failed: end the thread without waiting for a checkpoint that may never come."""
        self.work.put(("stop", 0, time.time()))

    def flush(self):
        while not self.msgs.empty():
            print(self.msgs.get())

    def _loop(self):
        while True:
            kind, n, t = self.work.get()
            if kind == "stop":
                self.say("training failed, nothing more to copy")
                return
            if kind == "final" and not os.path.isdir(self.src):
                self.say("no checkpoint_latest was written, nothing to copy")
                return
            tag = f"epoch{n}" if kind == "epoch" else "final"
            quiet = self.quiet_s if kind == "epoch" else 10
            try:
                for attempt in range(1, 4):
                    wait_stable(self.src, quiet_s=quiet, newer_than=(t - 2) if kind == "epoch" else None)
                    try:
                        files, size = copy_verified(self.src, self.dst, tag)
                    except SourceChanged:
                        self.say(f"{tag}: folder changed while copying, retry {attempt}/3")
                        continue
                    self.say(f"{tag}: quiet for {quiet}s -> copied to tmp -> verified {files} files / "
                             f"{size / 1e6:.0f} MB -> renamed to {os.path.basename(self.dst)}")
                    self.copies.append((tag, files, size))
                    break
                else:
                    raise RuntimeError(f"{tag}: source kept changing")
            except Exception as e:  # reported by the notebook; never kills training
                self.errors.append(f"{tag}: {e!r}")
                self.say(f"ERROR {tag}: {e!r}")
            if kind == "final":
                return


def run_logged(cmd, log_path, on_item=None):
    """Run cmd, echo its output live, save it to log_path. on_item((time, line)) sees every line; on_item(None) is a tick.

    Returns (return code, [(time, line), ...]).
    """
    q, events = queue.Queue(), []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    def reader():
        for line in proc.stdout:
            q.put((time.time(), line.rstrip("\n")))
        q.put(None)

    threading.Thread(target=reader, daemon=True).start()
    with open(log_path, "w", encoding="utf-8") as log:
        while True:
            try:
                item = q.get(timeout=1)
            except queue.Empty:
                if on_item:
                    on_item(None)
                continue
            if item is None:
                break
            events.append(item)
            log.write(item[1] + "\n")
            log.flush()
            print(item[1])
            if on_item:
                on_item(item)
    return proc.wait(), events


print("helpers ready")

helpers ready


In [13]:
# CELL 9: the 1-epoch training run (the pipeline is tested here, not the quality).
# laya-train writes to /content (local disk). A watcher copies checkpoint_latest/ to Drive only once it is fully written.
# EXPECT (about 3-8 min including model load):
#   "train items 19, calibration items 2, eval items 2 (calibration_slice) ... device cuda"
#   "epoch 1/1 mean loss ..."
#   a RuntimeWarning "... calibration: not fitted: 2 calibration items, fewer than 10 ..." (normal here: only 2 items)
#   "Training complete! Model saved to /content/work/smoke-ckpt"
#   watcher lines: "epoch1: quiet for 30s -> copied to tmp -> verified N files / X MB -> renamed ..." then "final: ..."
#   last line: "exit code 0 | ... | watcher errors: []"
OUT = f"{WORK}/smoke-ckpt"
shutil.rmtree(OUT, ignore_errors=True)
DRIVE_RUN = f"{DRIVE}/checkpoints/{RUN_ID}"
os.makedirs(DRIVE_RUN, exist_ok=True)

TRAIN_CMD = [sys.executable, "-u", LAUNCHER, "--data", SMOKE, "--base", BASE, "--out", OUT,
             "--loss", "soft-ce", "--epochs", "1", "--micro-batch", "4", "--grad-accum", "1", "--seed", "0",
             "--max-len", "512", "--head-max-len", "192"]
watcher = CkptWatcher(f"{OUT}/checkpoint_latest", f"{DRIVE_RUN}/checkpoint_latest", quiet_s=30)
t0 = time.time()
rc, events = run_logged(TRAIN_CMD, f"{WORK}/smoke_train.log",
                        on_item=lambda it: ((watcher.feed(it) if it else None), watcher.flush()))
(watcher.finish if rc == 0 else watcher.stop)()
watcher.thread.join(timeout=300)
watcher.flush()
print(f"exit code {rc} | wall {time.time() - t0:.0f}s | watcher copies: {len(watcher.copies)} | watcher errors: {watcher.errors}")
assert rc == 0, "laya-train itself failed: the error is in the output above, paste it"
assert not watcher.errors and watcher.copies, "training worked but the copy to Drive failed: paste the watcher lines"

train items 19, calibration items 2, eval items 2 (calibration_slice), skipped {}, device cuda:0
/usr/local/lib/python3.13/dist-packages/laya/train.py:694: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
epoch 1/1 mean loss 0.6490
/usr/local/lib/python3.13/dist-packages/laya/train_cli.py:221: RuntimeWarning: laya.train: noul calibration: not fitted: 2 calibration items, fewer than 10, so the temperature stays 1.0
  summary = finetune(

=== Calibration Evidence (in-sample calibration slice, 2 items; not held-out) ===
Metric                 Before         After          Delta         
-------------------------------------------------

In [14]:
# CELL 9b: TIMING RUN (nothing from it is kept). ~320 dummy rows padded to realistic lengths, 2 epochs,
# the real settings (micro-batch 8 x grad-accum 8). Labels are random: only the speed matters.
# It replaces the "1.5-2.5 h" guess for the full run with a measured number.
# EXPECT (about 3-8 min): "mean state tokens ~...", the epoch lines, then
#   "seconds per epoch: epoch 1 = ..., epoch 2 = ... (epoch 2 includes the epoch-1 checkpoint save)"
#   "estimate for 3,400 rows x 4 epochs: about X.X h" and a TIMING dict.
import random

import laya.common as lc
import laya.train as lt
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(f"{BASE}/tokenizer")
rng = random.Random(0)
FILLER = [
    "Agent: Thanks for bearing with me, I'm just pulling up the notes on your line now.",
    "Customer: No problem, take your time, I'm at my desk anyway.",
    "Agent: I can see the last technician visit was logged on the ninth, does that sound right?",
    "Customer: Yes, he came out in the afternoon and swapped the cable at the wall.",
    "Agent: Great, that helps. I'm checking whether the signal level changed after that visit.",
    "Customer: It was fine for a few days after he left and then it started dropping again.",
    "Agent: Understood. Let me run a quick line test from here, it takes about a minute.",
    "Customer: Sure, go ahead. The laptop is on the same Wi-Fi if you need it.",
    "Agent: The test shows some noise on the line in the evening hours, nothing at midday.",
    "Customer: That matches what I've seen, it's always worst after dinner.",
    "Agent: I'll note that on the ticket so the next team sees it without asking you again.",
    "Customer: Good, I really don't want to repeat all of this a third time.",
]
bases = [r for r in ROWS if "expected" in r]          # the 18 check rows


def padded(state, target):
    s = state
    for _ in range(60):
        if len(tok(s, add_special_tokens=False)["input_ids"]) >= target:
            break
        s += "\n" + rng.choice(FILLER)
    return s


timing_rows, lengths = [], []
for i in range(320):
    b = bases[i % len(bases)]
    q = b["questions"]["q"]
    room = lc.state_room(tok, lt.to_internal("q", q), 512, 192)      # state tokens that fit in the window
    state = padded(b["state"], int(room * rng.uniform(0.6, 0.97)))
    lengths.append(len(tok(state, add_special_tokens=False)["input_ids"]))
    timing_rows.append({"state": state, "questions": {"q": q}, "expected": {"q": rng.random() < 0.5}})
TIMING_DATA = f"{WORK}/timing.jsonl"
with open(TIMING_DATA, "w", encoding="utf-8", newline="\n") as f:
    f.write("\n".join(json.dumps(r, ensure_ascii=False) for r in timing_rows) + "\n")
print(f"320 rows | state tokens: mean {sum(lengths) / len(lengths):.0f}, min {min(lengths)}, max {max(lengths)} (window room about {room})")

T_OUT = f"{WORK}/timing-ckpt"
shutil.rmtree(T_OUT, ignore_errors=True)
t_cmd = [sys.executable, "-u", LAUNCHER, "--data", TIMING_DATA, "--base", BASE, "--out", T_OUT,
         "--loss", "soft-ce", "--epochs", "2", "--micro-batch", "8", "--grad-accum", "8", "--seed", "0",
         "--max-len", "512", "--head-max-len", "192"]
t0 = time.time()
rc, ev = run_logged(t_cmd, f"{WORK}/timing_train.log")
wall = time.time() - t0
assert rc == 0, "timing run failed: paste the output"

t_start = next(t for t, l in ev if l.startswith("train items"))
n_train = int(re.search(r"train items (\d+)", next(l for t, l in ev if l.startswith("train items"))).group(1))
t_e = [t for t, l in ev if EPOCH_RE.search(l)]
ep1, ep2 = t_e[0] - t_start, t_e[1] - t_e[0]
overhead = wall - ep1 - ep2
est_h = (ep2 / n_train * 0.9 * 3400 * 4 + overhead) / 3600
TIMING = {"rows": 320, "train_items": n_train, "mean_state_tokens": round(sum(lengths) / len(lengths)),
          "epoch1_s": round(ep1), "epoch2_s": round(ep2), "other_s": round(overhead),
          "est_hours_3400rows_4epochs": round(est_h, 2)}
print(f"seconds per epoch: epoch 1 = {ep1:.0f}, epoch 2 = {ep2:.0f} (epoch 2 includes the epoch-1 checkpoint save)")
print(f"load + calibration + final save = {overhead:.0f}s; items per second in epoch 2: {n_train / ep2:.1f}")
print(f"estimate for 3,400 rows x 4 epochs: about {est_h:.1f} h  (assumes similar row length; longer rows take longer)")
print("TIMING =", TIMING)
shutil.rmtree(T_OUT, ignore_errors=True)

320 rows | state tokens: mean 367, min 265, max 455 (window room about 425)
train items 288, calibration items 32, eval items 32 (calibration_slice), skipped {}, device cuda:0
epoch 1/2 mean loss 0.7874
epoch 2/2 mean loss 0.6630
/usr/local/lib/python3.13/dist-packages/laya/train_cli.py:221: RuntimeWarning: laya.train: noul calibration: fitted on only 32 calibration items
  summary = finetune(
/usr/local/lib/python3.13/dist-packages/laya/train_cli.py:221: RuntimeWarning: laya.train: noul calibration: the fitted temperature 5 is on the [0.5, 5] clamp
  summary = finetune(

=== Calibration Evidence (in-sample calibration slice, 32 items; not held-out) ===
Metric                 Before         After          Delta         
-----------------------------------------------------------------
Accuracy               0.5938         0.4375         -0.1563       
Loss                   0.6956         0.6980         +0.0024       
ECE                    0.2165         0.0712         -0.1453       


In [15]:
# CELL 10: what laya-train wrote.
# EXPECT: files incl. model.safetensors (about 842 MB, fp16), encoder/, tokenizer/, questions.json, train_report.json, checkpoint_latest/;
# "fine_tuned: True", "max_len: 512", "head_max_len: 192", "base temperature_by_options values still present: []"
# (the base model's old calibration buckets must NOT survive). With only 2 calibration items the new config
# normally has no temperature_by_options at all.
for dp, _, fns in sorted(os.walk(OUT)):
    if "checkpoint_latest" in dp:
        continue
    for fn in sorted(fns):
        p = os.path.join(dp, fn)
        print(f"{os.path.getsize(p) / 1e6:9.2f} MB  {os.path.relpath(p, OUT)}")
print("checkpoint_latest/ exists:", os.path.isdir(f"{OUT}/checkpoint_latest"))

cfg = json.load(open(f"{OUT}/rl_agent_config.json"))
print("fine_tuned:", cfg.get("fine_tuned"))
print("max_len:", cfg.get("max_len"), "| head_max_len:", cfg.get("head_max_len"))
print("temperature:", cfg.get("temperature"))
old = BASE_CFG.get("temperature_by_options", {})
new = cfg.get("temperature_by_options", {})
print("temperature_by_options in new config:", new)
print("base temperature_by_options values still present:", [k for k, v in new.items() if old.get(k) == v])
print("training keys:", sorted(cfg.get("training", {})))
print("calibration report:", json.dumps(cfg["training"].get("laya_train_calibration"), indent=1)[:700])

   842.61 MB  model.safetensors
     0.00 MB  questions.json
     0.00 MB  rl_agent_config.json
     0.00 MB  train_report.json
     0.00 MB  encoder/config.json
     3.58 MB  tokenizer/tokenizer.json
     0.00 MB  tokenizer/tokenizer_config.json
checkpoint_latest/ exists: True
fine_tuned: True
max_len: 512 | head_max_len: 192
temperature: [1.0, 1.0, 1.0]
temperature_by_options in new config: {}
base temperature_by_options values still present: []
training keys: ['epochs_completed', 'fine_tuned_from_checkpoint', 'hours', 'laya_train', 'laya_train_calibration', 'train_report', 'updates', 'world_size']
calibration report: {
 "choice": {
  "items": 0,
  "temperature": 1.0,
  "issues": []
 },
 "score": {
  "items": 0,
  "temperature": 1.0,
  "issues": []
 },
 "noul": {
  "items": 2,
  "temperature": 1.0,
  "issues": [
   "not fitted: 2 calibration items, fewer than 10, so the temperature stays 1.0"
  ],
  "accuracy": 0.5,
  "mean_confidence": 0.7287780323656701
 }
}


In [16]:
# CELL 11: reload the fine-tuned folder two ways and predict the 21 questions.
#   (a) laya.load(OUT)   (b) Router(models={"english": OUT}), which is how qa-analysis's laya_adapter will serve it.
# EXPECT: "max difference between the two load paths: 0.0000" (or ~1e-6), a table of base vs fine-tuned P(yes),
# and "moved toward their label: N/21". After only 5 training steps most rows should move toward the label,
# but small moves and a few wrong-way rows are normal. What must hold: no NaN and the two load paths agree.
import math

from laya import Router

ft_agent = laya.load(OUT)
P_FT = predict_all(lambda s, q: ft_agent.predict(s, q), ROWS)
del ft_agent
torch.cuda.empty_cache()

router = Router(models={"english": OUT}, default="english", max_loaded=1)
P_RT = predict_all(lambda s, q: router.predict(s, q, model="english"), ROWS)
del router
torch.cuda.empty_cache()

assert not any(math.isnan(v) for v in P_FT.values()), "NaN in fine-tuned predictions"
print(f"max difference between the two load paths: {max(abs(P_FT[k] - P_RT[k]) for k in P_FT):.6f}")
json.dump(P_FT, open(f"{WORK}/smoke_preds_ft.json", "w"), indent=1)

targets = {}
for r in ROWS:
    for qid in r["questions"]:
        key = f"{r['meta']['id']}/{qid}"
        targets[key] = float(r["expected"][qid]) if "expected" in r else r["gold"][qid]["probabilities"]["true"]
moved = 0
print(f"{'row/question':<28} {'target':>6} {'base':>7} {'tuned':>7}")
for k in P_FT:
    toward = abs(P_FT[k] - targets[k]) < abs(P_BASE[k] - targets[k])
    moved += toward
    print(f"{k:<28} {targets[k]:6.2f} {P_BASE[k]:7.3f} {P_FT[k]:7.3f} {'+' if toward else '-'}")
print(f"moved toward their label: {moved}/{len(P_FT)}")

max difference between the two load paths: 0.000000
row/question                 target    base   tuned
rapport_fail/q                 1.00   0.266   0.790 +
rapport_pass/q                 0.00   0.186   0.152 +
para_pass/q                    1.00   0.803   0.992 +
para_fail/q                    0.00   0.182   0.006 +
al_fail/q                      1.00   0.474   0.994 +
al_pass/q                      0.00   0.492   0.003 +
resolved_pass/q                1.00   0.751   0.859 +
resolved_fail/q                0.00   0.708   0.376 +
empathy_pass/q                 1.00   0.879   0.972 +
empathy_fail/q                 0.00   0.728   0.785 -
ownership_fail/q               1.00   0.737   0.993 +
ownership_pass/q               0.00   0.251   0.356 -
probing_pass/q                 1.00   0.776   0.888 +
probing_fail/q                 0.00   0.570   0.116 +
hostility_fail/q               1.00   0.324   0.306 -
hostility_pass/q               0.00   0.492   0.570 -
escalation_fail/q              1

In [17]:
# CELL 12: save everything to Drive (temp folder -> verify -> rename), then LOAD the Drive copies to prove they work.
# EXPECT:
#   "final copy: N files / X MB verified", "weights sha256 on Drive matches /content: True"
#   "Drive final loads; max diff vs cell 11: 0.0000..." (must be below 1e-4)
#   "Drive checkpoint_latest loads; max diff vs final: ..." (informational, should be tiny)
#   a manifest summary and the folder to download for the local CPU check.
DRIVE_FINAL = f"{DRIVE_RUN}/final"
wait_stable(OUT, quiet_s=5, ignore=("checkpoint_latest",))
files, size = copy_verified(OUT, DRIVE_FINAL, "final", ignore=("checkpoint_latest",))
print(f"final copy: {files} files / {size / 1e6:.0f} MB verified")
ok_hash = sha256_file(f"{OUT}/model.safetensors") == sha256_file(f"{DRIVE_FINAL}/model.safetensors")
print("weights sha256 on Drive matches /content:", ok_hash)
assert ok_hash

os.makedirs(f"{DRIVE_RUN}/files", exist_ok=True)
for name in ("smoke.jsonl", "smoke_preds_base.json", "smoke_preds_ft.json", "smoke_train.log", "timing_train.log"):
    if os.path.exists(f"{WORK}/{name}"):
        shutil.copy(f"{WORK}/{name}", f"{DRIVE_RUN}/files/{name}")

from importlib.metadata import version

manifest = {
    "run_id": RUN_ID,
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "purpose": "Plan A smoke test (plumbing only, not a quality result)",
    "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip(),
    "versions": {"python": sys.version.split()[0], "torch": version("torch"), "transformers": version("transformers"),
                 "laya": version("laya"), "laya_commit": LAYA_COMMIT},
    "base": {"repo": BASE_REPO, "revision": BASE_REV, "sha256": BASE_SHA256},
    "frozen_checks_sha256": FROZEN["sha256"],
    "qa_analysis_commit": FROZEN["qa_analysis_commit"],
    "smoke_jsonl_sha256": SMOKE_SHA256,
    "train_command": " ".join(TRAIN_CMD),
    "final_weights_sha256": sha256_file(f"{DRIVE_FINAL}/model.safetensors"),
    "watcher_copies": watcher.copies,
    "timing": globals().get("TIMING"),
}
json.dump(manifest, open(f"{DRIVE_RUN}/manifest.json", "w"), indent=2)
os.makedirs(f"{DRIVE}/runs/{RUN_ID}", exist_ok=True)
shutil.copy(f"{DRIVE_RUN}/manifest.json", f"{DRIVE}/runs/{RUN_ID}/manifest.json")

# load both Drive copies and compare with cell 11
drive_final = laya.load(DRIVE_FINAL)
P_DF = predict_all(lambda s, q: drive_final.predict(s, q), ROWS)
del drive_final
torch.cuda.empty_cache()
d1 = max(abs(P_DF[k] - P_FT[k]) for k in P_FT)
print(f"Drive final loads; max diff vs cell 11: {d1:.6f}")
assert d1 < 1e-4, "Drive copy predicts differently from /content: stop and tell Claude"

drive_latest = laya.load(f"{DRIVE_RUN}/checkpoint_latest")
P_DL = predict_all(lambda s, q: drive_latest.predict(s, q), ROWS)
del drive_latest
torch.cuda.empty_cache()
print(f"Drive checkpoint_latest loads; max diff vs final: {max(abs(P_DL[k] - P_DF[k]) for k in P_DF):.6f}")

print("\nleft in", DRIVE_RUN, ":", sorted(os.listdir(DRIVE_RUN)))
print("NEXT (on your PC): download", DRIVE_FINAL, "to laya-training/checkpoints/" + RUN_ID + "/final",
      "and", f"{DRIVE_RUN}/files/smoke_preds_ft.json", "to laya-training/checkpoints/" + RUN_ID + "/files/")
if globals().get("TIMING"):
    print("TIMING:", TIMING)

final copy: 7 files / 846 MB verified
weights sha256 on Drive matches /content: True
Drive final loads; max diff vs cell 11: 0.000000
Drive checkpoint_latest loads; max diff vs final: 0.000000

left in /content/drive/MyDrive/laya-finetuning/checkpoints/smoke-20261016-0848 : ['checkpoint_latest', 'files', 'final', 'manifest.json']
NEXT (on your PC): download /content/drive/MyDrive/laya-finetuning/checkpoints/smoke-20261016-0848/final to laya-training/checkpoints/smoke-20261016-0848/final and /content/drive/MyDrive/laya-finetuning/checkpoints/smoke-20261016-0848/files/smoke_preds_ft.json to laya-training/checkpoints/smoke-20261016-0848/files/
TIMING: {'rows': 320, 'train_items': 288, 'mean_state_tokens': 367, 'epoch1_s': 23, 'epoch2_s': 21, 'other_s': 16, 'est_hours_3400rows_4epochs': 0.25}


## Done on Colab

Paste back to Claude: the cell 9b `TIMING = {...}` line, and any cell whose output differs from its "EXPECT" comment.

Then, **on your PC** (not in Colab): download the `final` folder into `laya-training/checkpoints/<run id>/final` and `smoke_preds_ft.json` into `laya-training/checkpoints/<run id>/files/`, then run `tools/check_local_load.py` (see the README).